<a href="https://colab.research.google.com/github/johnsparz/multi_agent/blob/master/Multi_Agent.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Create Agents to Research and Write an Article



## 1. Install the modern dependencies


In [1]:
# Install the current stable CrewAI packages
!pip install -q -U crewai==1.15.2 crewai-tools==1.15.2

In [2]:
# Restart the runtime after installation if Colab asks you to.
# After restarting, run the notebook from the next cell onward.

import warnings
warnings.filterwarnings("ignore")

## 2. Configure the LLM


In [3]:
!pip install -U "crewai[litellm]"

In [4]:
import os
from crewai import LLM

llm = LLM(
    model="huggingface/meta-llama/Llama-3.1-8B-Instruct",
    api_key=os.getenv("HUGGINGFACE_API_KEY"),
    base_url="https://router.huggingface.co/v1",
    is_litellm=True
)

print("LLM initialized successfully!")

LLM initialized successfully!


In [10]:
import os
from getpass import getpass
from crewai import LLM

HUGGINGFACE_API_KEY = getpass("Enter your Hugging Face API key: ")

llm = LLM(
    model="huggingface/meta-llama/Llama-3.1-8B-Instruct",
    api_key=HUGGINGFACE_API_KEY,
    base_url="https://router.huggingface.co/v1"
)

Enter your Hugging Face API key: ··········


## 3. Create the Agents

We will build three specialized agents:

1. **Content Planner** — researches the topic conceptually and creates a structured plan.
2. **Content Writer** — turns the plan into a complete article.
3. **Editor** — reviews and improves the final article.

Each agent receives a `role`, `goal`, and `backstory`.

In [11]:
from crewai import Agent, Task, Crew, Process

In [12]:
# Agent 1: Content Planner
planner = Agent(
    role="Content Planner",
    goal="Plan engaging and factually accurate content on {topic}.",
    backstory=(
        "You are an experienced content planner. "
        "You create clear, useful, and well-structured content plans "
        "that help writers produce accurate and engaging articles. "
        "You identify the audience, key ideas, important context, "
        "and a logical structure for the article."
    ),
    llm=llm,
    allow_delegation=False,
    verbose=True,
)

In [13]:
# Agent 2: Content Writer
writer = Agent(
    role="Content Writer",
    goal="Write an insightful and factually responsible article about {topic}.",
    backstory=(
        "You are a skilled content writer who transforms structured content plans "
        "into clear, engaging articles. You write for a broad audience, explain "
        "complex ideas in accessible language, and distinguish objective information "
        "from opinions or interpretations."
    ),
    llm=llm,
    allow_delegation=False,
    verbose=True,
)

In [14]:
# Agent 3: Editor
editor = Agent(
    role="Editor",
    goal="Improve the article's clarity, structure, grammar, balance, and readability.",
    backstory=(
        "You are a meticulous editor. You review articles for grammar, clarity, "
        "logical flow, readability, balanced language, and consistency. "
        "You preserve useful ideas while removing repetition, unsupported claims, "
        "and awkward phrasing."
    ),
    llm=llm,
    allow_delegation=False,
    verbose=True,
)

## 4. Create the Tasks

Each task is assigned to one agent.

Because the tasks depend on one another, we will execute them sequentially:

**Plan → Write → Edit**

In [15]:
# Task 1: Plan
plan = Task(
    description=(
        "Create a detailed content plan for an article about {topic}.\n\n"
        "Your plan must include:\n"
        "1. The target audience and their likely interests.\n"
        "2. The main ideas and key points that should be covered.\n"
        "3. A logical article outline with an introduction, body sections, and conclusion.\n"
        "4. Important concepts, examples, or context the writer should include.\n"
        "5. Suggested keywords and useful angles for the article."
    ),
    expected_output=(
        "A comprehensive content plan containing audience analysis, "
        "key points, a structured article outline, suggested keywords, "
        "and important context for the writer."
    ),
    agent=planner,
)

In [16]:
# Task 2: Write
write = Task(
    description=(
        "Using the Content Planner's work, write a complete article about {topic}.\n\n"
        "Requirements:\n"
        "1. Use the content plan as the foundation of the article.\n"
        "2. Use clear and engaging headings.\n"
        "3. Explain technical or complex concepts in accessible language.\n"
        "4. Incorporate relevant keywords naturally.\n"
        "5. Include an engaging introduction, a well-developed body, and a conclusion.\n"
        "6. Avoid unsupported certainty and clearly distinguish facts from opinions.\n"
        "7. Return the article in clean Markdown format."
    ),
    expected_output=(
        "A complete, publication-ready article in Markdown format with "
        "a strong introduction, organized sections, coherent paragraphs, "
        "and a clear conclusion."
    ),
    agent=writer,
)

In [17]:
# Task 3: Edit
edit = Task(
    description=(
        "Review the article produced by the Content Writer. "
        "Correct grammar, spelling, awkward phrasing, repetition, "
        "structural problems, and unclear claims. "
        "Improve the article's flow and readability while preserving "
        "its core meaning. Make sure the final version is professional "
        "and suitable for publication."
    ),
    expected_output=(
        "A polished, publication-ready Markdown article with improved "
        "grammar, clarity, structure, flow, and readability."
    ),
    agent=editor,
)

## 5. Create the Crew

`Process.sequential` ensures that the tasks run in order:

1. The planner creates the content plan.
2. The writer receives the previous task's output and writes the article.
3. The editor receives the article and produces the polished final version.

In [18]:
crew = Crew(
    agents=[planner, writer, editor],
    tasks=[plan, write, edit],
    process=Process.sequential,
    verbose=True,
)

## 6. Run the Crew

We will start with the topic **Artificial Intelligence**.

The final result should be the edited article produced by the last task.

In [20]:
result = crew.kickoff_async(
    inputs={"topic": "Artificial Intelligence"}
)

print("Crew execution completed.")

Crew execution completed.


In [ ]:
from IPython.display import Markdown, display

display(Markdown(result.raw))

## 7. Try Your Own Topic

Change the topic below and run the cells.

In [24]:
from IPython.display import Markdown, display

topic = "The Future of Generative AI"

result = await crew.kickoff_async(
    inputs={"topic": topic}
)

display(Markdown(result.raw))

╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: 795e1387-88a0-4714-af55-b451ccfff454                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: Create a detailed content plan for an article about The Future of Generative AI.                         │
│                                                                                                                 │
│  Your plan must include:                                                                                        │
│  1. The target audience and their likely interests.                                                             │
│  2. The main ideas and key points that should be covered.                                                       │
│  3. A logical article outline with an introduction, body sections, and conclusion.                              │
│  4. Important concepts, examples, or context the writer should include.                                         │
│  5. Suggested keywords and useful angles for the article.                                                       │
│  ID: f558d305-0d4d-4ddb-9707-acc5af933306                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Content Planner                                                                                         │
│                                                                                                                 │
│  Task: Create a detailed content plan for an article about The Future of Generative AI.                         │
│                                                                                                                 │
│  Your plan must include:                                                                                        │
│  1. The target audience and their likely interests.                                                             │
│  2. The main ideas and key points that should be covered.                                                       │
│  3. A logical article outline with an introduction, body sections, and conclusion.                              │
│  4. Important concepts, examples, or context the writer should include.                                         │
│  5. Suggested keywords and useful angles for the article.                                                       │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

[Finalize] todos_count=0, todos_with_results=0


╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Content Planner                                                                                         │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  **Content Plan: The Future of Generative AI**                                                                  │
│                                                                                                                 │
│  **1. Target Audience and Likely Interests**                                                                    │
│                                                                                                                 │
│  The target audience for this article is technology enthusiasts, business leaders, and individuals interested   │
│  in the advancements and potential applications of artificial intelligence (AI). They likely:                   │
│                                                                                                                 │
│  - Follow the latest developments in AI research and its applications                                           │
│  - Are curious about the potential impact of generative AI on various industries                                │
│  - Want to stay informed about the future of AI and its potential disruptions                                   │
│  - Are interested in the possibilities and limitations of generative AI                                         │
│                                                                                                                 │
│  **2. Main Ideas and Key Points**                                                                               │
│                                                                                                                 │
│  The article should cover the following key points:                                                             │
│                                                                                                                 │
│  - Introduction to generative AI and its current applications                                                   │
│  - The potential impact of generative AI on various industries, such as art, media, and entertainment           │
│  - The role of generative AI in business and its potential use cases                                            │
│  - The challenges and limitations of generative AI, including ethics, bias, and safety concerns                 │
│  - Future developments and predictions in the field of generative AI                                            │
│                                                                                                                 │
│  **3. Logical Article Outline**                                                                                 │
│                                                                                                                 │
│  I. **Introduction**                                                                                            │
│                                                                                                                 │
│  * Definition of generative AI and its current applications                                                     │
│  * Importance of generative AI in the present and future                                                        │
│  * Brief overview of the article's content                                                                      │
│                                                        

...

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Editor                                                                                                  │
│                                                                                                                 │
│  Task: Review the article produced by the Content Writer. Correct grammar, spelling, awkward phrasing,          │
│  repetition, structural problems, and unclear claims. Improve the article's flow and readability while          │
│  preserving its core meaning. Make sure the final version is professional and suitable for publication.         │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

[Finalize] todos_count=0, todos_with_results=0


╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Editor                                                                                                  │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  **The Future of Generative AI: Unlocking Creative Potential and Transforming Industries**                      │
│                                                                                                                 │
│  **Introduction**                                                                                               │
│                                                                                                                 │
│  Generative artificial intelligence (AI) has revolutionized the way we think about creativity and innovation.   │
│  By harnessing the power of machine learning and neural networks, generative AI has enabled the creation of     │
│  complex and captivating content, from art and music to literature and video. In this article, we'll explore    │
│  the current applications of generative AI, its potential impact on various industries, and the challenges and  │
│  limitations of this rapidly evolving technology. We'll also delve into the future developments and             │
│  predictions that will shape the landscape of generative AI.                                                    │
│                                                                                                                 │
│  **The Rise of Generative AI in Art and Media**                                                                 │
│                                                                                                                 │
│  Generative AI has already made a significant mark in the art world, with AI-generated works challenging        │
│  traditional notions of creativity and authorship. AI-generated paintings, sculptures, music, and literature    │
│  are pushing the boundaries of what is possible in creative industries. Notable examples include the            │
│  AI-generated artwork "Edmond de Belamy," which sold for $432,500 at Christie's auction house in 2018, and      │
│  Amper Music's AI-powered composition tool, which can generate custom music tracks in minutes. Additionally,    │
│  AI-generated literature, such as the AI-written novel "The Day a Computer Writes a Novel," is also gaining     │
│  attention.                                                                                                     │
│                                                                                                                 │
│  The potential impact of generative AI on the creative industries is significant, with AI-generated content     │
│  potentially disrupting traditional business models and creating new opportunities for artists and creators.    │
│  However, this also raises concerns about authorship, ownership, and the value of creative labor. As            │
│  generative AI becomes more sophisticated, it's essential to establish clear guidelines and regulations for     │
│  AI-generated content to ensure fair compensation and recognition for human creators.                           │
│                                                                                                                 │
│  **Generative AI in Business: Opportunities and Use Cases**                                                     │
│                                                                                                                 │
│  Generative AI is not limited to the creative industrie

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name: Review the article produced by the Content Writer. Correct grammar, spelling, awkward phrasing,          │
│  repetition, structural problems, and unclear claims. Improve the article's flow and readability while          │
│  preserving its core meaning. Make sure the final version is professional and suitable for publication.         │
│  Agent: Editor                                                                                                  │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

**The Future of Generative AI: Unlocking Creative Potential and Transforming Industries**

**Introduction**

Generative artificial intelligence (AI) has revolutionized the way we think about creativity and innovation. By harnessing the power of machine learning and neural networks, generative AI has enabled the creation of complex and captivating content, from art and music to literature and video. In this article, we'll explore the current applications of generative AI, its potential impact on various industries, and the challenges and limitations of this rapidly evolving technology. We'll also delve into the future developments and predictions that will shape the landscape of generative AI.

**The Rise of Generative AI in Art and Media**

Generative AI has already made a significant mark in the art world, with AI-generated works challenging traditional notions of creativity and authorship. AI-generated paintings, sculptures, music, and literature are pushing the boundaries of what is possible in creative industries. Notable examples include the AI-generated artwork "Edmond de Belamy," which sold for $432,500 at Christie's auction house in 2018, and Amper Music's AI-powered composition tool, which can generate custom music tracks in minutes. Additionally, AI-generated literature, such as the AI-written novel "The Day a Computer Writes a Novel," is also gaining attention.

The potential impact of generative AI on the creative industries is significant, with AI-generated content potentially disrupting traditional business models and creating new opportunities for artists and creators. However, this also raises concerns about authorship, ownership, and the value of creative labor. As generative AI becomes more sophisticated, it's essential to establish clear guidelines and regulations for AI-generated content to ensure fair compensation and recognition for human creators.

**Generative AI in Business: Opportunities and Use Cases**

Generative AI is not limited to the creative industries; it has also found its way into the business world, where it's being used to improve operations and decision-making. Companies like Google and Microsoft are already utilizing generative AI to generate personalized product recommendations, automate customer service, and even create new business models. For instance, Google's AI-powered tool, AutoML, enables users to create custom machine learning models without extensive coding knowledge.

The benefits of implementing generative AI in the workplace are numerous, including improved efficiency, increased productivity, and enhanced customer experiences. However, there are also challenges to consider, such as the need for human oversight and the potential for bias in AI-generated content. To mitigate these risks, companies must ensure that AI systems are designed and trained to minimize bias and ensure fairness.

**Challenges and Limitations of Generative AI**

While generative AI holds much promise, it also raises important concerns about ethics, bias, and safety. AI-generated deepfakes, which are AI-generated videos or images that mimic real individuals, have raised concerns about the potential for deception and manipulation. Other challenges include ensuring transparency and accountability in AI-generated content, as well as mitigating the risk of bias and prejudice.

To address these challenges, it's essential to develop and deploy generative AI responsibly, with a focus on transparency, accountability, and inclusivity. This includes implementing guidelines and regulations for AI-generated content, as well as ensuring that AI systems are designed and trained to minimize bias and ensure fairness.

**The Future of Generative AI: Predictions and Developments**

As research and development in generative AI continue to advance, we can expect to see even more innovative applications and advancements in the field. Some predictions include:

* Increased use of generative AI in creative industries, including art, music, and literature
* Widespread adoption of generative AI in business, including improved operations and decision-making
* Development of more sophisticated AI systems that can learn and adapt to complex tasks and environments
* Growing concerns about ethics, bias, and safety in AI-generated content

In conclusion, generative AI has the potential to unlock creative potential and transform industries, but it also raises important concerns about ethics, bias, and safety. As we move forward, it's essential to develop and deploy generative AI responsibly, with a focus on transparency, accountability, and inclusivity. By doing so, we can unlock the full potential of generative AI and create a brighter, more innovative future for all.

**Key Takeaways**

* Generative AI has revolutionized the way we think about creativity and innovation, with applications in art, media, and business.
* The rise of generative AI in art and media has sparked a heated debate about authorship, ownership, and the value of creative labor.
* Generative AI has the potential to improve operations and decision-making in business, but also raises concerns about bias and safety.
* Responsible development and deployment of generative AI is essential to address ethics, bias, and safety concerns.
* Future developments and predictions in generative AI include increased use in creative industries, widespread adoption in business, and growing concerns about ethics and safety.

## Summary

This notebook now uses the modern CrewAI workflow:

- `crewai==1.15.2`
- `crewai-tools==1.15.2`
- CrewAI's native `LLM` interface
- `Agent`
- `Task`
- `Crew`
- `Process.sequential`
- `crew.kickoff(inputs={...})`

The original learning objective is preserved: multiple specialized agents collaborate to plan, write, and edit an article.